# 06 - Batch Inference using SageMaker

This notebook performs batch inference using the trained Logistic Regression model.

It includes:
- Loading a pre-trained model artifact from S3 (registered or tar.gz)
- Specifying a batch input CSV (unlabeled data)
- Running a SageMaker batch transform job
- Saving and optionally uploading the predictions
- Notes and guards for local development (this file is written locally but designed to run inside SageMaker)


## Sagemaker Setup

In [ ]:
import boto3
import sagemaker
from sagemaker import image_uris
from datetime import datetime

# Setup SageMaker session and roles
session = sagemaker.Session()
role = sagemaker.get_execution_role()
bucket = session.default_bucket()
region = session.boto_region_name

# Timestamp for naming
timestamp = datetime.now().strftime("%Y-%m-%d-%H-%M-%S")

# Paths
model_artifact_uri = f"s3://{bucket}/diabetes/registry/lr_model.tar.gz"
batch_input_uri = f"s3://{bucket}/diabetes/batch/new_unseen_data.csv"
batch_output_uri = f"s3://{bucket}/diabetes/batch/output/{timestamp}/"

print("Model Artifact:", model_artifact_uri)
print("Input Data:", batch_input_uri)
print("Output Location:", batch_output_uri)


## Create Sagemaker Model

In [2]:
from sagemaker.model import Model
from sagemaker import image_uris

sklearn_image_uri = image_uris.retrieve("sklearn", region=region, version="1.0-1")

model = Model(
    model_data=model_artifact_uri,
    image_uri=sklearn_image_uri,
    role=role,
    name=f"logistic-batch-model-{timestamp}",
    sagemaker_session=session
)


## Get Validation Data 

In [4]:
import pandas as pd
import pickle
import boto3
import os

# Path setup
local_path = "data/X_val.pkl"
s3_key = "diabetes/data/X_val.pkl"

LOAD_MODE = 'local'

if LOAD_MODE == "s3":
    print("Loading X_val from S3...")
    s3 = boto3.client("s3")
    os.makedirs("data", exist_ok=True)  # Ensure local folder exists
    with open(local_path, "wb") as f:
        s3.download_fileobj(bucket, s3_key, f)

# Load from local file (works for both local and S3 modes)
with open(local_path, "rb") as f:
    X_val = pickle.load(f)

# Create smaller file for faster execution
X_val = X_val.head(100)
print("Loaded X_val. Shape:", X_val.shape)


Loaded X_val. Shape: (100, 189)


In [5]:
import os

# Save to CSV
os.makedirs("data", exist_ok=True)
X_val.to_csv("data/X_val.csv", index=False)

# Upload to S3 in /batch/ folder
batch_input_uri = session.upload_data(
    path="data/X_val.csv",
    bucket=bucket,
    key_prefix="diabetes/batch"
)

print("Uploaded X_val.csv for batch inference.")
print("S3 path:", batch_input_uri)


Uploaded X_val.csv for batch inference.
S3 path: s3://sagemaker-us-east-1-380537322556/diabetes/batch/X_val.csv


In [6]:
# Upload to S3

batch_input_uri = session.upload_data(
    path="data/X_val.csv",
    bucket=bucket,
    key_prefix="diabetes/batch"
)


## Batch Inference Setup

In [7]:
from datetime import datetime

# Timestamp to organize batch outputs
timestamp = datetime.now().strftime("%Y-%m-%d-%H-%M-%S")
batch_output_uri = f"s3://{bucket}/diabetes/batch/output/{timestamp}/"
model_artifact_uri = f"s3://{bucket}/diabetes/registry/lr_model.tar.gz"

print("Output path will be:", batch_output_uri)


Output path will be: s3://sagemaker-us-east-1-380537322556/diabetes/batch/output/2025-06-20-17-53-24/


In [ ]:
import contextlib

transformer = model.transformer(
    instance_count=1,
    instance_type="ml.m5.large",
    assemble_with="Line",             
    output_path=batch_output_uri,
    accept="text/csv"
)

# Run batch transform and suppress streaming logs
with open("batch_transform_log.txt", "w") as f, contextlib.redirect_stdout(f):
    transformer.transform(
        data=batch_input_uri,
        content_type="text/csv",
        split_type="Line",
        wait=True
    )

print("Batch transform complete.")
print("Predictions saved to:", batch_output_uri)
print("Logs written to 'batch_transform_log.txt'")


INFO:sagemaker:Creating model with name: logistic-batch-model-2025-06-20-17-52-13
INFO:sagemaker:Creating transform job with name: logistic-batch-model-2025-06-20-17-52-1-2025-06-20-18-02-03-092


## Inspect Predictions

In [ ]:
import re
import pandas as pd

def download_batch_output(s3_uri):
    match = re.match(r"s3://([^/]+)/(.+)", s3_uri)
    bucket_name, prefix = match.group(1), match.group(2)

    s3 = boto3.client("s3")
    response = s3.list_objects_v2(Bucket=bucket_name, Prefix=prefix)
    output_file = next(obj["Key"] for obj in response["Contents"] if obj["Key"].endswith(".out"))

    s3.download_file(bucket_name, output_file, "predictions.out")
    return pd.read_csv("predictions.out", header=None)

df_preds = download_batch_output(batch_output_uri)
df_preds.head()
